# H-D3 — Windowed-attention hybrid vs pure MinGRU (needle recall)

**Claim.** The backbone bake-off (H-D1) chose MinGRU on *bpc*, which is blind to
content-based retrieval. On an induction task the toy showed pure MinGRU fails at
distance while a windowed-attention hybrid succeeds *at bounded state*. This
notebook tests it at real scale: two runs identical except `CB_BACKBONE`, scored
on needle recall across lengths.

**Read the needle table, not the training log** — both arms look similar on bpc /
generations; the difference lives only in recall at distance. The windowed hybrid
should recall where the needle is within its window, and fall back to MinGRU
beyond it (the honest bounded-state ceiling).

In [ ]:
# --- setup: clone repo, deps, mount Drive (run once per session) ---
import os, subprocess, sys, time
if not os.path.exists('/content/CubbyLLM'):
    !git clone -q https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM
else:
    !cd /content/CubbyLLM && git pull -q --ff-only
!pip -q install torch numpy sentencepiece
from google.colab import drive; drive.mount('/content/drive')
REPO = '/content/CubbyLLM'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- EDIT to your Drive locations ---
DRIVE     = '/content/drive/MyDrive/cubbyllm'
TOKENIZER = f'{DRIVE}/grillcheese_bbpe128k.json'   # your tokenizer .json/.model
CORPUS_DRIVE = f'{DRIVE}/token_cache'              # uint32 shards on Drive
CORPUS    = '/content/token_cache'                 # staged to LOCAL SSD

# Stage the corpus to local SSD. Drive-FUSE memmap reads bottleneck the GPU —
# this run measured 3,000 tok/s off Drive vs ~100,000 tok/s off local disk.
# Skip (comment out) if CORPUS is already populated this session.
!mkdir -p {CORPUS} && rsync -a --info=progress2 {CORPUS_DRIVE}/ {CORPUS}/
!du -sh {CORPUS}

In [ ]:
# --- shared config: identical for both arms except CB_BACKBONE ---
BASE = dict(
    CB_D='512', CB_L='8', CB_HEADS='8',          # 512/8=64 per head (even -> RoPE ok)
    CB_ATTN_EVERY='3', CB_WINDOW='512',          # hybrid-only; mingru ignores them
    CB_B='32', CB_S='1024', CB_STEPS='30000',    # 0.98B tok/arm, ~9.8 tok/param
    CB_LR='3e-4', CB_WARMUP='2000',              # real-corpus LR + warmup (3e-3 diverges)
    CB_EVAL='20', CB_GRAD_CKPT='1',
    CB_HEALTH='500', CB_GEN='500', CB_GEN_PROMPT='1',
)

In [ ]:
# --- run a script, tee to a log, catch a hung child on interrupt ---
def run(script, env_extra, log_name):
    env = dict(os.environ, CUBBY_SPM=TOKENIZER, CB_CORPUS=CORPUS, **env_extra)
    os.makedirs(f'{REPO}/validation/logs', exist_ok=True)
    log = f'{REPO}/validation/logs/{log_name}'
    p = None
    try:
        with open(log, 'a', encoding='utf-8', buffering=1) as f:
            f.write(f"\n=== {time.strftime('%F %T')} "
                    + ' '.join(f'{k}={v}' for k, v in sorted(env_extra.items())) + '\n')
            p = subprocess.Popen([sys.executable, '-u', f'validation/{script}'],
                                 cwd=REPO, env=env, stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in p.stdout:
                print(line, end=''); f.write(line)
            p.wait()
    finally:
        if p and p.poll() is None:      # never leave a child holding the GPU
            p.terminate()

In [ ]:
# --- ARM 1: pure MinGRU baseline (~2.6h) ---
run('train_colab.py',
    dict(BASE, CB_BACKBONE='mingru', CB_CKPT=f'{DRIVE}/ab_mingru.pt'),
    'ab_mingru.log')

In [ ]:
# --- ARM 2: windowed hybrid — one flag different (~2.6h) ---
run('train_colab.py',
    dict(BASE, CB_BACKBONE='hybrid', CB_CKPT=f'{DRIVE}/ab_hybrid.pt'),
    'ab_hybrid.log')

In [ ]:
# --- NEEDLE EVAL: both checkpoints, lengths spanning the window ---
for name in ('mingru', 'hybrid'):
    print(f'\n########## NEEDLE: {name} ##########', flush=True)
    run('exp_needle_recall.py',
        dict(CB_CKPT=f'{DRIVE}/ab_{name}.pt', CB_LENGTHS='256,512,1024,4096'),
        f'needle_{name}.log')

### How to read

Each arm prints **TRAINED** vs two controls (**shuffled** needle, **untrained**
model), PMI-scored, with an argmax-spread check. Compare TRAINED against its own
shuffled control — never against 100%.

- **Hybrid recalls where MinGRU is at its shuffled floor** (short lengths / shallow
  depths, needle within ~window of the query) → the deployable win. H-D3 goes
  from toy-VERIFIED to scale-VERIFIED, and the hybrid earns becoming the default.
- **Both fall to chance at 4096-deep** (needle far beyond the window) → the honest
  bounded-state ceiling, the limit the 1M-context claim must respect.
- **No separation anywhere** → the toy didn't transfer; pure MinGRU stands.